# M02 · Figures and maps

**Part 0 · week 1 · about 6 h core + 1.5 h stretch**

## 한국어 요약

이 모듈에서는 matplotlib의 그림(Figure)과 축(Axes) 구조를 익히고, 연구실 그림 규칙(첫 단어만 대문자로 쓰는 축 이름, 미국식 철자, Helvetica 글꼴, 네 변이 모두 있는 테두리)에 맞춰 그림을 그립니다. 지도 기호(원)의 지름에서 규모를 거꾸로 구하는 함수(연구실 기호 크기 규칙의 역함수)와 연도별 지진 수를 세는 함수를 직접 작성하고, 2019년 상주 지진 시퀀스의 시간-규모 그림을 그립니다. 이어서 PyGMT로 상주 지진과 반경 100 km 안의 관측소를 축척 막대가 있는 지도에 표시합니다. 그림에 나타나는 숫자는 모두 코드로 출력해서 직접 확인하는 습관을 기릅니다.

## What you will be able to do

- Write a vectorized function that reads a map marker diameter back as a local magnitude (the inverse of the house rule)
- Count earthquakes per UTC year with every year present, zeros included, as a pandas Series
- Make a time-magnitude figure in the lab house style and print every number it shows
- Draw a PyGMT map of the Sangju sequence with magnitude-sized circles, labeled stations within 100 km and a scale bar
- Explain how the yearly counts around the Sangju mainshock change with the search radius (stretch)

## Before you start

- Modules: M01
- Reading: Plotting and Programming in Python, episode Plotting; PyGMT tutorials, Making your first figure; Plotting data points
- Needs: catalog
- Key terms (see the glossary): function (함수), vectorization (벡터화), magnitude (규모), local magnitude (ML; 국지 규모), earthquake catalog (지진 목록, 지진 카탈로그), DataFrame (데이터프레임), origin time (진원시, 발생 시각), UTC (협정 세계시), KST (한국 표준시), KOGL Type 1 (공공누리 제1유형: 출처표시), earthquake sequence (지진 시퀀스), mainshock (본진), aftershock (여진), station (관측소), network code (관측망 코드), great-circle distance (대원 거리), head wave (선두파), map projection (지도 투영법), epicenter (진앙), seismicity (지진활동), histogram (히스토그램), scatter plot (산점도), figure and axes (그림과 축 영역), sentence case (첫 단어만 대문자로 쓰는 표기), time-magnitude plot (시간-규모 그림), map region (지도 범위), scale bar (축척 막대), decimal degrees (십진 도 표기)

## Passing this module

Each task is done when its `st.check(...)` cell prints PASS. The pass criteria are frozen in [criteria/M02.yaml](../../../.sgtl/criteria/M02.yaml).

## AI use

Follow [docs/ai_policy.md](../../../docs/ai_policy.md) and log every AI conversation in `work/ai_log/M02.md`.

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Other cells read them from here.
SEQUENCE = "sangju2019"                    # a named sequence from st.data.sequence
YEARS = (2018, 2022)                       # first and last year for the yearly counts (UTC)
STATION_RADIUS_KM = 100.0                  # show the stations within this distance of the sequence center (km)
MAP_REGION = [126.8, 129.4, 35.5, 37.5]    # map region: lon_min, lon_max, lat_min, lat_max (degrees)
MAP_SCALE_KM = 50                          # length of the scale bar on the map (km)
NEAR_RADIUS_KM = 20.0                      # stretch task: radius around the mainshock (km)
NEAR_YEARS = (2000, 2025)                  # stretch task: first and last year (UTC)
FIGURE_DIR = "figures"                     # your figure files go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M02")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pygmt  # import PyGMT after st.init, which points it to this environment's GMT library

Path(FIGURE_DIR).mkdir(exist_ok=True)

## How this module works

A figure is often the first thing a reader looks at, and the place where a mistake hides most easily. This
module teaches the two plotting tools of the lab and its **house style**, the short list of rules that every
figure here follows:

- **Matplotlib**, a Python library for line plots, scatter plots and histograms;
- **PyGMT**, Python access to GMT (the Generic Mapping Tools), which draws maps.

You will write four short **functions** (named blocks of code that take inputs and return an output; glossary
`function`), two more in the stretch task, and use them on the 2019 Sangju earthquakes. Every task has the same
steps: read the task, write a prediction, write the function, run the check, explain what you found. The
checks test a few properties; your mentor looks at how your figures read.

One habit runs through the whole module: **print every number your figure shows.** If a figure shows 27
events between 2018 and 2022 with magnitudes up to 3.9, a cell next to it prints those numbers. Then you and
your readers can check the picture against the data instead of trusting it.

Words in **bold** are explained where they first appear, and most are in the glossary:

In [ ]:
print(st.glossary.term("function"))
print(st.glossary.term("vectorization"))

## 1. A figure and its axes (Matplotlib)

Matplotlib builds every plot from a few objects:

- a **figure** is the whole picture, the thing you save to a file;
- an **axes** is one plotting area inside the figure, with an x axis, a y axis, labels and a title; one figure
  can hold several axes side by side;
- everything drawn on an axes (a line, a set of markers, a piece of text) is an **artist**.

`fig, ax = plt.subplots()` makes one figure with one axes. You draw by calling methods of `ax` (`ax.plot`,
`ax.scatter`, `ax.hist`) and label with `ax.set_xlabel`, `ax.set_ylabel` and `ax.set_title`. A tiny example
with made-up numbers:

In [ ]:
# Made-up numbers: four days of water levels (not seismic data).
day = [1, 2, 3, 4]
level_m = [2.0, 3.5, 3.0, 3.2]

fig, ax = plt.subplots(figsize=(4, 2.5))  # one figure with one axes; the size is in inches
ax.plot(day, level_m, marker="o")         # a line plot with a marker at each point
ax.set_xlabel("Day of the experiment")
ax.set_ylabel("Water level (m)")
ax.set_title("A toy line plot")

# Print every number the figure shows:
print("axes in this figure:", len(fig.axes), "| x range:", ax.get_xlim(), "| highest level:", max(level_m), "m")

Three kinds of plots cover most of this week:

- a **line plot** (`ax.plot`) joins the points in order; use it when the order means something, as in time;
- a **scatter plot** (`ax.scatter`) draws one marker per point and no joining line;
- a **histogram** (`ax.hist`) splits values into **bins** (intervals such as 1.0 to 1.1) and draws one bar per
  bin, as tall as the number of values in it.

The practice magnitudes below come from `st.synth`, which makes up data with known properties; they are not
real earthquakes.

In [ ]:
rng = np.random.default_rng(2)                            # a fixed seed: the same numbers on every run
toy_m = st.synth.gr_magnitudes(rng, 300, b=1.0, mc=1.0)   # 300 made-up magnitudes, all 1.0 or larger
edges = np.arange(0.95, toy_m.max() + 0.1, 0.1)           # bin edges halfway between the 0.1 steps

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 2.8))     # one figure, two axes side by side
ax1.scatter(np.arange(toy_m.size), toy_m, s=6)            # s is the marker AREA in points squared
ax1.set_xlabel("Event number")
ax1.set_ylabel(st.plot.ML_LABEL)
heights, _, _ = ax2.hist(toy_m, bins=edges, color="0.6", edgecolor="black")
ax2.set_xlabel(st.plot.ML_LABEL)
ax2.set_ylabel("Number of events")
fig.tight_layout()                                        # keeps labels of neighboring axes apart

# Print every number the figure shows:
print(f"{toy_m.size} magnitudes from {toy_m.min():.1f} to {toy_m.max():.1f}")
print(f"tallest bar: {heights.max():.0f} events in the bin from {edges[heights.argmax()]:.2f}")

### The house style

Every figure in the lab follows the same rules, so that figures from different people fit together:

1. **Sentence case** for labels, titles and legend entries: capitalize only the first word and names. Write
   "Origin time (UTC)", not "Origin Time (UTC)" and not "ORIGIN TIME".
2. **American English** in figures: color, center, gray, meter, hypocenter.
3. **Helvetica** as the font. `st.init` already called `st.plot.use_house_style()`; it prints a note when
   Helvetica is missing and a similar font stands in. Call it again if you ever reset Matplotlib.
4. **All four frame lines** around each axes; do not hide the top and right lines.
5. Units in parentheses, as in "Depth (km)".
6. **Short titles**, 40 characters at most: the title names the figure, the caption explains it.

**Magnitude** (glossary `magnitude`) is a number on a logarithmic scale for the size of an earthquake: one
unit more means about ten times larger amplitudes. The KMA catalog gives the **local magnitude**
(glossary `local_magnitude`), computed from the largest amplitudes on nearby seismometers and corrected for
distance. It is written $\mathit{M}_{\mathrm{L}}$, an italic M with a roman subscript L, and
`st.plot.ML_LABEL` holds exactly that.

In [ ]:
print("fonts, in order of preference:", plt.rcParams["font.sans-serif"][:3])
print("frame lines drawn:", [s for s in ("left", "right", "top", "bottom") if plt.rcParams[f"axes.spines.{s}"]])
print("magnitude label:", st.plot.ML_LABEL)

## 2. Magnitude as marker size, and back

On a map a bigger earthquake gets a bigger circle. The lab uses one rule for the diameter, so maps from
different people can be compared. An earthquake of local magnitude $M$ is drawn as a circle of diameter

$$d = 0.025\,\sqrt{5\,e^{2M}}\ \mathrm{cm},$$

where $e \approx 2.718$ is Euler's number ($e^{x}$ is `np.exp(x)` in NumPy). `st.plot.mag_size_cm(M)` applies
this rule for you, and your map in section 5 uses it. Because magnitude is logarithmic, the rule grows
exponentially: every extra magnitude unit multiplies the diameter by the same factor.

A reader of a map goes the other way: from the width of a circle to the magnitude. That needs the **inverse**
of the rule, a calculation that undoes it, the way division undoes multiplication. The step that undoes
$e^{x}$ is the **natural logarithm** $\ln$: $\ln x$ is the power you must raise $e$ to in order to get $x$, so
$\ln(e^{y}) = y$ for every $y$. NumPy writes it `np.log`. Do not confuse it with `np.log10`, the logarithm to
base 10, which undoes $10^{y}$ instead:

In [ ]:
y = np.array([-1.0, 0.0, 2.5])
print("y                 ->", y)
print("np.log(np.exp(y)) ->", np.log(np.exp(y)))        # the natural logarithm undoes e to a power
print("np.log10(10.0**y) ->", np.log10(10.0 ** y))      # the base-10 logarithm undoes 10 to a power
print("np.exp(np.log(4)) ->", np.exp(np.log(4.0)))      # and e to a power undoes the natural logarithm

Your function must work on one number and on a whole column of values. That is **vectorization**: doing a
calculation on a whole array at once instead of looping over the items. NumPy functions such as `np.exp`,
`np.log` and `np.sqrt` are vectorized; Python lists are not:

In [ ]:
a = np.array([1.0, 2.0, 3.0])
print("2 * array     ->", 2 * a)                     # element by element
print("np.exp(array) ->", np.exp(a))                 # element by element
print("2 * list      ->", 2 * [1.0, 2.0, 3.0])       # a list is REPEATED, not multiplied
print("np.asarray(list) * 2 ->", np.asarray([1.0, 2.0, 3.0]) * 2)

One unit trap before you start: Matplotlib's `ax.scatter(..., s=...)` takes the marker **area** in points
squared (one point is 1/72 inch), while PyGMT takes the **diameter**, here in centimeters. The house rule
gives a diameter in cm; for Matplotlib you would convert it with `s = (d_cm / 2.54 * 72) ** 2`.

### M02-T1 · Magnitude from marker size · core · about 30 min

**Goal:** write `ml_from_marker_cm(size_cm)`, which reads circles on a map backwards: from a marker diameter
to the local magnitude it stands for.

**Given:** the house rule above, $d = 0.025\,\sqrt{5\,e^{2M}}$ cm, and the natural logarithm `np.log`. Solve
the rule for $M$ on paper first, one operation at a time, before you write any code.

**Contract:**
- Input `size_cm`: one number, a Python list, a NumPy array or a pandas Series of marker diameters in
  centimeters, all larger than 0.
- Output: local magnitudes (no unit), one per diameter: a float for one number; a NumPy array (or a pandas
  Series) of the same length for a list, an array or a Series.
- The exact inverse of the house rule: the rule applied to your magnitude gives back the diameter you started
  from. Circles smaller than the one for $\mathit{M}_{\mathrm{L}}$ 0 stand for magnitudes below 0.
- Vectorized: no Python loop over the values.
- Do not call `st.plot.mag_size_cm` or anything else from `sgtl_tutorial` inside the function: the inverse is
  yours to work out.

**Done when** `st.check("M02-T1", func=ml_from_marker_cm)` passes. The check makes up fresh magnitudes, turns
them into diameters with the house rule and asks your function for the magnitudes back (220 cases; each one
within 1e-9 magnitude units).

**Stuck?** `st.hint("M02-T1", 1)`

In [ ]:
# Before you code: which magnitude does a 1-cm circle stand for, and by how much does the magnitude change when a
# circle gets twice as wide?
st.predict("M02-T1", """

""")

In [ ]:
def ml_from_marker_cm(size_cm):
    """Local magnitude(s) whose house-rule circle has the diameter size_cm (cm): the inverse of the house rule.

    size_cm : a number, a list, a NumPy array or a pandas Series of diameters in cm (all larger than 0).
    Returns a float for one number, otherwise a NumPy array (or Series) of the same length.
    """
    # Step 1: turn the input into a NumPy array of floats (a list cannot do element-by-element math).
    # Step 2: on paper, solve d = 0.025 * sqrt(5 * e^(2M)) for M, undoing one operation at a time.
    # Step 3: write your result once, with NumPy functions, for the whole array.
    # Step 4: give back a plain float when a single number came in.
    # YOUR CODE HERE
    raise NotImplementedError("M02-T1: write your code here")

In [ ]:
st.check("M02-T1", func=ml_from_marker_cm)

**Explain (3-5 sentences):** Why does the house rule let the diameter grow like $e^{M}$ instead of in
proportion to $M$? Why does your inverse need `np.log` and not `np.log10`, and which magnitude would
`np.log10` have given for a 1-cm circle? What goes wrong when a plain Python list reaches your function
without being converted first?

In [ ]:
# Use it: read a magnitude key backwards, then go forward again with the house rule.
key_cm = np.array([0.25, 0.5, 1.0, 2.0, 3.0])          # circle diameters on a map key (cm)
key_ml = ml_from_marker_cm(key_cm)
for d_cm, m in zip(key_cm, key_ml):
    print(f"a {d_cm:.2f}-cm circle stands for M_L {m:.2f}")
back_cm = st.plot.mag_size_cm(key_ml)
print(f"largest difference after the round trip: {np.max(np.abs(back_cm - key_cm)):.1e} cm")

## 3. Counting events per year

An **earthquake catalog** (glossary `earthquake_catalog`) is a table of earthquakes with origin time,
location, depth and magnitude, made by a seismological agency. `st.data.kma_catalog()` loads the KMA catalog
subset that comes with this tutorial as a pandas **DataFrame** (glossary `dataframe`: a table with named
columns). The **origin time** (glossary `origin_time`: when the rupture starts) appears twice: `origin_utc` in
**UTC** (Coordinated Universal Time, the clock for everything in this tutorial) and `origin_kst` in **KST**
(Korea Standard Time, UTC plus 9 hours). An event at 20:00 UTC on 31 December is already 05:00 on 1 January
in Korea, so the two clocks can disagree about the year.

The KMA catalog may be reused under **KOGL Type 1** (glossary `kogl`), the Korea Open Government License: anyone
may reuse the data if they name the source. Every figure made from it names KMA as the source.

Our example is the **2019 Sangju sequence**. An **earthquake sequence** (glossary `earthquake_sequence`) is a
group of earthquakes close in space and time that are thought to be related; its largest event is the
**mainshock**, and the smaller events that follow it nearby are **aftershocks**. `st.data.sequence("sangju2019")`
gives the events within 10 km of the sequence center from 2018 to 2022.

In [ ]:
sangju = st.data.sequence(SEQUENCE)
print(sangju.attrs["source"])
print(len(sangju), "events; columns:", list(sangju.columns))
sangju.head(3)

A pandas **Series** is one column of values with labels, and the labels are its **index**. Yearly counts fit
a Series whose index is the year. A made-up example:

In [ ]:
toy_counts = pd.Series([3, 1, 4], index=[2019, 2021, 2022])   # made-up counts
print(toy_counts)
print("index:", list(toy_counts.index), "| value for 2021:", toy_counts.loc[2021])
print("Is 2020 in the index?", 2020 in toy_counts.index)

In `toy_counts` the year 2020 is simply absent. A rate, a mean or a bar chart computed from it would skip 2020
without any warning, as if the year had never happened. A quiet year is information: it must appear, with 0.

### M02-T2 · Events per year, zeros included · core · about 45 min

**Goal:** write `yearly_counts(df, start_year, end_year)`, the number of events in each year.

**Given:** a catalog DataFrame with the column `origin_utc` (time-zone-aware UTC times), such as `sangju`.

**Contract:**
- Count events by the **UTC** calendar year of `origin_utc`.
- Return a pandas Series of integer counts whose index is every year from `start_year` to `end_year`, both
  included, as plain integers in increasing order.
- A year without events appears with 0; it is not left out.
- Events before `start_year` or after `end_year` are not counted.
- Do not change `df`.

**Done when** `st.check("M02-T2", func=yearly_counts)` passes. The check builds made-up catalogs with known
yearly counts: quiet years, busy years just outside the range and events late on 31 December (UTC).

**Stuck?** `st.hint("M02-T2", 1)`

In [ ]:
# Before you code: how many Sangju events do you expect in each year from 2018 to 2022, and what should your
# function return for a year without events?
st.predict("M02-T2", """

""")

In [ ]:
def yearly_counts(df, start_year, end_year):
    """Number of events per UTC calendar year, for every year start_year..end_year (both included).

    df : catalog DataFrame with a time-zone-aware column 'origin_utc'.
    Returns a pandas Series of integers indexed by the years (int), zeros included.
    """
    # Step 1: the UTC year of every event.
    # Step 2: how many events each year has.
    # Step 3: put the counts onto the full list of years, with 0 for a year without events.
    # YOUR CODE HERE
    raise NotImplementedError("M02-T2: write your code here")

In [ ]:
st.check("M02-T2", func=yearly_counts)

**Explain (3-5 sentences):** Why is a missing year different from a year with 0 events when you later compute
a rate or draw a bar chart? Which events would move to another year if you used `origin_kst`, and how many
Sangju events is that (print it)? Did your counts match your prediction?

In [ ]:
# Use it: the Sangju counts, printed and drawn as a bar chart.
counts = yearly_counts(sangju, *YEARS)
print(counts.to_string())
print("total:", int(counts.sum()), "of", len(sangju), "events in the sequence")

fig, ax = plt.subplots(figsize=(4.5, 2.6))
ax.bar(counts.index, counts.to_numpy(), color="0.6", edgecolor="black")
ax.set_xticks(counts.index)
ax.set_xlabel("Year (UTC)")
ax.set_ylabel("Number of events")
ax.set_title("Sangju sequence per year")
fig.tight_layout()

## 4. A time-magnitude figure

A **time-magnitude figure** puts each event at its origin time (x) and its magnitude (y). It shows at a glance
when a sequence started, how big its largest events were and how the activity changed afterwards.

Matplotlib understands datetime columns: give it a column of times as x and it picks date ticks by itself.
The tick labels are in UTC (Matplotlib's default time zone), so the x label should say "(UTC)". A tiny
example with made-up daily values:

In [ ]:
toy_days = pd.date_range("2021-03-01", periods=10, freq="D", tz="UTC")   # ten made-up days
toy_values = [3, 5, 4, 6, 8, 7, 7, 9, 6, 5]
fig, ax = plt.subplots(figsize=(4.5, 2.4))
ax.plot(toy_days, toy_values)
ax.set_xlabel("Date (UTC)")
ax.set_ylabel("Made-up value")
fig.autofmt_xdate()                       # tilts the date labels so they do not overlap
print("x data type:", toy_days.dtype, "| first and last day:", toy_days[0].date(), toy_days[-1].date())

A function that makes a figure should **return the Figure object**. Then the caller decides what to do with
it: show it, change it, or save it with `fig.savefig("name.png")` (and `.pdf` for print). Saving inside the
function, or calling `plt.show()` there, takes that choice away.

### M02-T3 · Time-magnitude figure · core · about 75 min

**Goal:** write `time_magnitude_figure(df, title)`, which draws every event of a catalog at its origin time
and magnitude and returns the figure.

**Given:** a catalog DataFrame with `origin_utc` (UTC) and `ml`, and a short title.

**Contract:**
- Return a Matplotlib `Figure` (not the axes, not `None`) with at least one axes.
- x: the origin times as dates (the `origin_utc` column itself, not event numbers or decimal years);
  y: the local magnitude; one marker (or stem) per event.
- Labels in sentence case and American English; the x label names the time zone; the y label names the
  magnitude with a capital M (`st.plot.ML_LABEL`).
- Show `title` as the title; every title at most 40 characters.
- Keep all four frame lines. Do not call `plt.show()` inside the function.

**Done when** `st.check("M02-T3", func=time_magnitude_figure)` passes. The check draws two made-up catalogs
with your function and tests only the properties above. Whether the figure reads well (marker size, colors,
clutter, date labels) is judged by your mentor with the rubric.

**Stuck?** `st.hint("M02-T3", 1)`

In [ ]:
# Before you code: where will the tallest marker of the Sangju figure sit, and how will the other events spread
# over 2018-2022?
st.predict("M02-T3", """

""")

In [ ]:
def time_magnitude_figure(df, title):
    """Magnitude against origin time, one marker per event; returns the Matplotlib Figure.

    df : catalog DataFrame with 'origin_utc' (time-zone-aware UTC) and 'ml'.
    title : a short title in sentence case (at most 40 characters).
    """
    # Step 1: one figure with one axes.
    # Step 2: one marker per event at (origin time, magnitude).
    # Step 3: labels (sentence case, time zone, magnitude symbol) and the title; return the figure.
    # YOUR CODE HERE
    raise NotImplementedError("M02-T3: write your code here")

In [ ]:
st.check("M02-T3", func=time_magnitude_figure)

**Explain (3-5 sentences):** Which numbers does your Sangju figure show (number of events, time span, largest
magnitude), and where did you print them? Why must the x label name the time zone? If you plotted
`origin_kst` instead of `origin_utc`, would the markers move? Try it and explain what you see.

In [ ]:
# Use it: the Sangju figure, saved as PNG and PDF.
fig = time_magnitude_figure(sangju, "Sangju sequence, 2018-2022")
for ext in ("png", "pdf"):
    fig.savefig(Path(FIGURE_DIR) / f"sangju_time_magnitude.{ext}")

# Print every number the figure shows:
big = sangju.loc[sangju["ml"].idxmax()]
first, last = sangju["origin_utc"].min(), sangju["origin_utc"].max()
print(f"{len(sangju)} events from {first:%Y-%m-%d} to {last:%Y-%m-%d} (UTC)")
print(f"M_L from {sangju['ml'].min():.1f} to {sangju['ml'].max():.1f}; "
      f"largest: event {big['event_id']} at {big['origin_utc']:%Y-%m-%d %H:%M:%S} UTC")
print("Source:", sangju.attrs["source"])

## 5. Maps with PyGMT

**GMT** draws a map in layers, one command per layer, and **PyGMT** calls those commands from Python.
Everything goes onto one `pygmt.Figure`:

- `fig.basemap(region=..., projection=..., frame=...)` sets the map area and draws the frame. The **region** is
  `[lon_min, lon_max, lat_min, lat_max]` in degrees. The projection is the **map projection** (glossary
  `map_projection`): the rule that flattens the curved Earth onto paper. `"M12c"` is a Mercator projection
  12 cm wide, fine for an area the size of Korea. `frame=["af", "+tA title"]` labels the axes automatically
  and adds a title.
- `fig.coast(...)` draws coastlines, land and water.
- `fig.plot(x=..., y=..., style=...)` draws symbols. A style is a letter and a size: `"c0.3c"` is a circle
  0.3 cm wide, `"t0.3c"` a triangle, `"a0.5c"` a star. With `style="cc"` (circle, sizes in cm) the sizes come
  from `size=...`, one value per point.
- `fig.text(x=..., y=..., text=..., justify=..., offset=...)` writes text. `justify="LM"` puts the left middle
  of the text at the point and `offset="0.2c/0c"` moves it 0.2 cm to the right, so a label sits **beside** its
  symbol instead of on top of it.
- `pygmt.makecpt(...)` makes a color table for a range of values, `fig.colorbar(...)` explains it, and
  `fig.legend()` lists the symbols that were drawn with `label=...`.
- `map_scale=...` (in `fig.basemap` or `fig.coast`) draws a **scale bar**, a bar of known length in km.
  Every lab map has one; `st.plot.scale_bar_spec(region, km=...)` writes its settings.

The house style for maps: degrees as decimal numbers (36.50°, not 36°30'), a plain frame (a thin line instead
of GMT's checkered border), Helvetica, a short title and always a scale bar. Everything drawn inside
`with st.plot.gmt_style():` gets the first three. Layers are painted in order, and a later layer covers an
earlier one, so the scale bar comes after the land. A toy map with nothing on it yet:

In [ ]:
demo_region = [124.0, 132.0, 33.0, 39.0]                     # lon_min, lon_max, lat_min, lat_max (degrees)
with st.plot.gmt_style(FONT_TITLE="12p,Helvetica"):          # house style, plus a smaller title font
    fig = pygmt.Figure()
    fig.basemap(region=demo_region, projection="M10c", frame=["af", "+tToy map"])
    fig.coast(land="gray90", water="white", shorelines="0.25p,gray30")
    fig.basemap(map_scale=st.plot.scale_bar_spec(demo_region, km=100))   # after the land, so it stays visible
fig.show()
print("region:", demo_region, "| scale bar settings:", st.plot.scale_bar_spec(demo_region, km=100))

Symbols, text, a color bar and a legend, again with made-up points (not earthquakes):

In [ ]:
toy = pd.DataFrame({"name": ["P1", "P2", "P3"], "lon": [127.0, 128.6, 126.5], "lat": [37.5, 35.2, 33.4],
                    "value": [5.0, 12.0, 20.0], "size_cm": [0.2, 0.4, 0.6]})
with st.plot.gmt_style(FONT_TITLE="12p,Helvetica"):
    fig = pygmt.Figure()
    fig.basemap(region=demo_region, projection="M10c", frame=["af", "+tToy symbols"])
    fig.coast(land="gray90", water="white", shorelines="0.25p,gray30")
    pygmt.makecpt(cmap="viridis", series=[0, 25])            # colors for values from 0 to 25
    fig.plot(x=toy["lon"], y=toy["lat"], size=toy["size_cm"], style="cc", fill=toy["value"], cmap=True,
             pen="0.5p,black")
    fig.colorbar(frame=["af", "x+lMade-up value"], position="JMR+o0.5c/0c+w4c")
    fig.plot(x=toy["lon"], y=toy["lat"], style="x0.15c", pen="0.8p,red", label="Toy point")
    fig.text(x=toy["lon"], y=toy["lat"], text=toy["name"], justify="LM", offset="0.4c/0c", font="8p,Helvetica")
    fig.legend(position="jTL+o0.2c", box="+gwhite+p0.5p")
    fig.basemap(map_scale=st.plot.scale_bar_spec(demo_region, km=100))
fig.show()

# Print every number the map shows:
print(toy.to_string(index=False))

GMT writes special text with `@` codes: `@%2%M@%%` sets an M in Helvetica-Oblique (font number 2) and
`@-L@-` writes a subscript L, so `"@%2%M@%%@-L@- 3"` reads as $\mathit{M}_{\mathrm{L}}$ 3 on a map.

### Stations near the sequence

A **station** (glossary `station`) is a site with a seismometer, named by a **network code** (glossary
`network_code`: KS for KMA, KG for KIGAM) and a station code. `st.data.stations()` lists the stations of this
tutorial. The lab locates earthquakes with stations within about 100 km: farther away, the first wave to arrive
can be a **head wave** (glossary `head_wave`), a wave that ran along the base of the crust, and the lab never
uses those for locations. So the Sangju map shows the stations within 100 km. The cell below finds them with a
geodesy library (pyproj). Distances along the Earth's surface are **great-circle distances** (glossary
`great_circle_distance`), which you met in M01.

In [ ]:
from pyproj import Geod

center = sangju.attrs["sequence"]              # the sequence settings, including its center
stations_all = st.data.stations()
n_sta = len(stations_all)
_, _, dist_m = Geod(ellps="WGS84").inv(np.full(n_sta, center["center_lon"]), np.full(n_sta, center["center_lat"]),
                                       stations_all["longitude"].to_numpy(), stations_all["latitude"].to_numpy())
near_stations = stations_all.loc[dist_m / 1000.0 <= STATION_RADIUS_KM].reset_index(drop=True)
print(f"{len(near_stations)} of {n_sta} stations lie within {STATION_RADIUS_KM:g} km of "
      f"{center['center_lat']:.2f} N, {center['center_lon']:.2f} E")
print(near_stations["network"].value_counts().to_string())

### M02-T4 · Map of the Sangju sequence · core · about 100 min

**Goal:** write `map_events(df, stations, region, scale_km)`, which draws a lab-style map of earthquakes and
stations and returns it.

**Given:** the Sangju events (`sangju`), the stations within `STATION_RADIUS_KM` (`near_stations`), the map
region `MAP_REGION` and the scale-bar length `MAP_SCALE_KM` from the parameters cell, and the house rule
`st.plot.mag_size_cm` for the circle sizes.

**Contract:**
- `df`: a catalog DataFrame (`longitude` and `latitude` in degrees, `ml`, `depth_km`); `stations`: a DataFrame
  with `station`, `longitude`, `latitude`; `region`: `[lon_min, lon_max, lat_min, lat_max]` in degrees;
  `scale_km`: the scale-bar length in km.
- Return the `pygmt.Figure`; do not call `fig.show()` inside the function.
- Use exactly the given region.
- Events: circles from `fig.plot(x=..., y=..., size=..., style="cc", ...)`, where `size` holds one diameter in
  cm per event from the house rule, `st.plot.mag_size_cm`.
- Stations: a symbol for each (`fig.plot` with `x=` and `y=`) and its code written beside it (`fig.text` with
  `text=`).
- A scale bar `scale_km` long (`map_scale=...`), visible on top of the land.
- The house style: draw inside `with st.plot.gmt_style():`, with a short title (40 characters at most).

**Done when** `st.check("M02-T4", func=map_events, region=MAP_REGION, scale_km=MAP_SCALE_KM)` passes. The check
draws a made-up map and the Sangju map with your function and tests only the properties above. How the map
looks (colors, clutter, where labels sit, the legend) is judged by your mentor with the rubric.

**Stuck?** `st.hint("M02-T4", 1)`

In [ ]:
# Before you code: how big will the Sangju cluster and its mainshock circle look on the map, and roughly how many
# stations do you expect within 100 km?
st.predict("M02-T4", """

""")

In [ ]:
def map_events(df, stations, region, scale_km):
    """PyGMT map of earthquakes (circles sized by magnitude) and stations (labeled symbols).

    df : catalog DataFrame with longitude, latitude (degrees), ml and depth_km.
    stations : DataFrame with station, longitude, latitude.
    region : [lon_min, lon_max, lat_min, lat_max] in degrees.
    scale_km : length of the scale bar in km.
    Returns the pygmt.Figure.
    """
    # Step 1: inside "with st.plot.gmt_style():", a new figure with a frame for exactly `region` and a short title.
    # Step 2: the events as circles (style "cc"), sized by one house-rule diameter in cm per event.
    # Step 3: every station as a symbol, with its code written beside it.
    # Step 4: a scale bar scale_km long, drawn after the land; return the figure (no fig.show() inside).
    # YOUR CODE HERE
    raise NotImplementedError("M02-T4: write your code here")

In [ ]:
st.check("M02-T4", func=map_events, region=MAP_REGION, scale_km=MAP_SCALE_KM)

**Explain (3-5 sentences):** Which numbers does your map show (region, scale-bar length, magnitude range,
circle sizes, number of stations), and where did you print them? Why must the scale bar come after the land?
Which stations would you choose to locate the Sangju events, and why does the lab stay within about 100 km?

In [ ]:
# Use it: the Sangju map, shown and saved.
fig = map_events(sangju, near_stations, MAP_REGION, MAP_SCALE_KM)
fig.show()
for ext in ("png", "pdf"):
    fig.savefig(Path(FIGURE_DIR) / f"sangju_map.{ext}")

# Print every number the map shows:
sizes_cm = np.sort(np.asarray(st.plot.mag_size_cm(sangju["ml"]), dtype=float))
print("region (lon_min, lon_max, lat_min, lat_max):", MAP_REGION, "| scale bar:", MAP_SCALE_KM, "km")
print(f"{len(sangju)} events, M_L {sangju['ml'].min():.1f} to {sangju['ml'].max():.1f}, "
      f"depth {sangju['depth_km'].min():.0f} to {sangju['depth_km'].max():.0f} km")
print(f"circle diameters from {sizes_cm[0]:.2f} to {sizes_cm[-1]:.2f} cm; read back with your inverse: "
      f"M_L {ml_from_marker_cm(sizes_cm[0]):.1f} to {ml_from_marker_cm(sizes_cm[-1]):.1f}")
print(len(near_stations), "stations within", STATION_RADIUS_KM, "km | source:", sangju.attrs["source"])

## 6. Stretch: how many events near the mainshock?

`st.data.sequence` cut the Sangju sequence out with a fixed 10-km circle. Is that a good circle? Count the
events in a circle of your own radius around the mainshock, year by year, in the whole catalog. The counts mix
the sequence with the **seismicity** (glossary `seismicity`: the earthquakes of a region over a period of
time) that was there before it; your figure will show which is which.

### M02-T5 · Yearly counts near the mainshock · stretch · about 90 min

**Goal:** write `events_near(df, lat0, lon0, radius_km)` (the rows within `radius_km` of a point) and
`counts_near(df, lat0, lon0, radius_km, start_year, end_year)` (their yearly counts), then draw a
time-magnitude figure of the events within `NEAR_RADIUS_KM` of the Sangju mainshock.

**Given:** the whole catalog `st.data.kma_catalog()`, the mainshock (the largest event of `sangju`), and your
`yearly_counts` and `time_magnitude_figure`.

**Contract:**
- Distance: the great-circle distance in km from `(lat0, lon0)` (degrees) to each **epicenter** (glossary
  `epicenter`: the point on the surface above where the earthquake started). A sphere of radius 6371 km is
  accurate enough: the check does not judge events within 0.5 % of the radius, where a sphere and the real,
  slightly flattened Earth can disagree.
- Keep an event when its distance is at most `radius_km`.
- `counts_near` returns the same kind of Series as `yearly_counts`: every year from `start_year` to
  `end_year`, zeros included, integer counts by UTC year.
- Do not use `sgtl_tutorial` inside the functions.

**Done when** `st.check("M02-T5", func=counts_near)` passes. The check uses a made-up catalog with events at
known distances, then the real catalog around the Sangju mainshock with a fresh radius between 10 and 30 km;
it never prints the real counts.

**Stuck?** `st.hint("M02-T5", 1)`

In [ ]:
# Before you code: with NEAR_RADIUS_KM, how will the yearly counts change from 2000 to 2025, and will 2019
# stand out?
st.predict("M02-T5", """

""")

In [ ]:
def events_near(df, lat0, lon0, radius_km):
    """Rows of df whose epicenter lies within radius_km (km) of the point (lat0, lon0) in degrees."""
    # YOUR CODE HERE
    raise NotImplementedError("M02-T5: write your code here")


def counts_near(df, lat0, lon0, radius_km, start_year, end_year):
    """Events per UTC year within radius_km of (lat0, lon0), for every year start_year..end_year, zeros included."""
    # YOUR CODE HERE
    raise NotImplementedError("M02-T5: write your code here")

In [ ]:
st.check("M02-T5", func=counts_near)

**Explain (3-5 sentences):** How do the yearly counts change when you set `NEAR_RADIUS_KM` to 10, 20 and 30 km
(print them)? Which part of the counts belongs to the 2019 sequence and which to the background seismicity?
Is the 10-km circle of `st.data.sequence` a good choice, and what would you check before trusting the counts
of the early years?

In [ ]:
# Use it: counts and a time-magnitude figure around the mainshock.
catalog = st.data.kma_catalog()
main = sangju.loc[sangju["ml"].idxmax()]
print(f"Mainshock {main['event_id']}: {main['origin_utc']:%Y-%m-%d %H:%M:%S} UTC, M_L {main['ml']:.1f}, "
      f"{main['latitude']:.2f} N, {main['longitude']:.2f} E, depth {main['depth_km']:.0f} km")
near_counts = counts_near(catalog, main["latitude"], main["longitude"], NEAR_RADIUS_KM, *NEAR_YEARS)
print(near_counts.to_string())

near = events_near(catalog, main["latitude"], main["longitude"], NEAR_RADIUS_KM)
t0 = pd.Timestamp(year=NEAR_YEARS[0], month=1, day=1, tz="UTC")
t1 = pd.Timestamp(year=NEAR_YEARS[1] + 1, month=1, day=1, tz="UTC")
near = near[(near["origin_utc"] >= t0) & (near["origin_utc"] < t1)]
fig = time_magnitude_figure(near, f"Within {NEAR_RADIUS_KM:g} km of the mainshock")
for ext in ("png", "pdf"):
    fig.savefig(Path(FIGURE_DIR) / f"sangju_within_{NEAR_RADIUS_KM:g}km.{ext}")
print(f"{len(near)} events, M_L {near['ml'].min():.1f} to {near['ml'].max():.1f} | source: {catalog.attrs['source']}")

## Before you finish

Your mentor reviews your figures with this list (the "correctness" and "craft" items of the rubric in
`docs/how_this_tutorial_works.md`):

- Labels and titles in sentence case and American English, units in parentheses, the time zone named.
- Helvetica (or the noted stand-in) and all four frame lines.
- Maps: decimal degrees, a plain frame, a scale bar, a short title, circle sizes from the house rule, station
  codes beside their symbols, small events visible on top of big ones.
- Every number a figure shows is printed in a cell next to it.
- Figures made from KMA data name the source (KMA, KOGL Type 1).

Your figure files stay in `figures/` inside your work folder. Commit your notebook as you learned in M00, and
log your AI conversations in `work/ai_log/M02.md`.

In [ ]:
st.progress("M02")